# Import required libraries

In [6]:
import os
import re
import string
import random
from collections import defaultdict, Counter
import math
from math import log, exp


# Download and Process the IMDB Dataset

In [ ]:

def load_imdb_unsup_sentences(folder_path):

    all_sentences = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as f:
                text = f.read()
                # Replace <br /> with <nl> token
                text = text.replace("<br />", " <nl> ")
                sentences = text.split('\n')
                all_sentences.extend([s.strip() for s in sentences if s.strip()])
    return all_sentences

def remove_punctuation(text):

    # Create a translation table that maps all punctuation to None, except '<' and '>'
    punct = ''.join(c for c in string.punctuation if c not in '<>')
    trans_table = str.maketrans('', '', punct)
    return text.translate(trans_table)

def build_vocabulary(sentences):

    vocab = set()
    for sentence in sentences:
        sentence = sentence.lower()
        sentence = remove_punctuation(sentence)
        tokens = sentence.split()
        vocab.update(tokens)
    return vocab

def tokinize(sentences, vocab, unknown="<UNK>"):

    tokenized_sentences = []
    for sentence in sentences:
        sentence = sentence.lower()
        sentence = remove_punctuation(sentence)
        tokens = sentence.split()
        # Replace tokens not in vocabulary with unknown token
        tokens = [token if token in vocab else unknown for token in tokens]
        tokenized_sentences.append(tokens)
    return tokenized_sentences

# Load dataset

In [8]:
imdb_folder = "imdb_data/unsup"
sentences = load_imdb_unsup_sentences(imdb_folder)

print(f"Number of raw sentences loaded: {len(sentences)}")
print(f"Example (first 2 sentences):\n{sentences[:2]}")


Number of raw sentences loaded: 50000
Example (first 2 sentences):
['I admit, the great majority of films released before say 1933 are just not for me. Of the dozen or so "major" silents I have viewed, one I loved (The Crowd), and two were very good (The Last Command and City Lights, that latter Chaplin circa 1931). <nl>  <nl> So I was apprehensive about this one, and humor is often difficult to appreciate (uh, enjoy) decades later. I did like the lead actors, but thought little of the film. <nl>  <nl> One intriguing sequence. Early on, the guys are supposed to get "de-loused" and for about three minutes, fully dressed, do some schtick. In the background, perhaps three dozen men pass by, all naked, white and black (WWI ?), and for most, their butts, part or full backside, are shown. Was this an early variation of beefcake courtesy of Howard Hughes?', 'Take a low budget, inexperienced actors doubling as production staff\x97 as well as limited facilities\x97and you can\'t expect much mor

In [9]:
assert len(sentences) == 50000, "Expected 50,000 sentences from the unsup folder."

# Train-Test Split

In [ ]:
random.seed(42)

def split_data(sentences, test_split=0.1):

    # Create a copy of sentences and shuffle it
    shuffled = sentences.copy()
    random.shuffle(shuffled)
    
    # Calculate split point
    split_idx = int(len(shuffled) * (1 - test_split))
    
    # Split into train and test
    train_sentences = shuffled[:split_idx]
    test_sentences = shuffled[split_idx:]
    
    return train_sentences, test_sentences


In [11]:
train_sentences, test_sentences = split_data(sentences)

print(f"Number of training sentences: {len(train_sentences)}")
print(f"Number of test sentences: {len(test_sentences)}")

Number of training sentences: 45000
Number of test sentences: 5000


In [12]:
assert len(train_sentences) == 45000, "Expected 45,000 sentences for training."
assert len(test_sentences) == 5000, "Expected 5,000 sentences for testing."


In [13]:
vocab = build_vocabulary(train_sentences)
tokinized_sentences = tokinize(train_sentences, vocab)

print(f"Vocabulary size: {len(vocab)}")
print(f"Example tokens from first sentence: {tokinized_sentences[0][:10] if tokinized_sentences else 'No tokens loaded'} ...")


Vocabulary size: 161366
Example tokens from first sentence: ['having', 'first', 'seen', 'the', 'directors', '12min', 'take', 'on', 'poes', 'fall'] ...


In [14]:
# assert len(vocab) == 161292, "Expected a vocabulary size of 171,591." #skip for replication problems
assert len(tokinized_sentences) == 45000, "Expected tokenized sentences count to match raw sentences."

example = "I love Natural language processing, and i want to be a great engineer."
assert len(example) == 70, "Example sentence length (in characters) does not match the expected 70."

example_tokens = tokinize([example], vocab)[0]
assert len(example_tokens) == 13, "Token count for the example sentence does not match the expected 13."


# Build ngram counts

In [ ]:

def pad_sentence(tokens, n):
    """
    Pads a list of tokens with <s> at the start (n-1 times)
    and </s> at the end (once).
    For example, if n=3, you add 2 <s> tokens at the start.
    """
    return ['<s>'] * (n-1) + tokens + ['</s>']

def build_ngram_counts(tokenized_sentences, n):

    ngram_counts = Counter()
    context_counts = Counter()
    
    for sentence in tokenized_sentences:
        padded = pad_sentence(sentence, n)
        
        # Count n-grams
        for i in range(len(padded) - n + 1):
            ngram = tuple(padded[i:i+n])
            context = tuple(padded[i:i+n-1])
            ngram_counts[ngram] += 1
            context_counts[context] += 1
    
    return ngram_counts, context_counts


# Probability function

In [ ]:

def laplace_probability(ngram, ngram_counts, context_counts, vocab_size, alpha=1.0):
    """
    Computes the probability of an n-gram using Laplace (add-alpha) smoothing.

    P(w_i | w_{i-(n-1)}, ..., w_{i-1}) =
        (count(ngram) + alpha) / (count(context) + alpha * vocab_size)

    Args:
        ngram: tuple of tokens representing the n-gram
        ngram_counts: Counter of n-grams
        context_counts: Counter of (n-1)-gram contexts
        vocab_size: size of the vocabulary
        alpha: smoothing parameter (1.0 = add-1 smoothing)

    Returns:
        Probability of the given n-gram.
    """
    context = ngram[:-1]
    numerator = ngram_counts[ngram] + alpha
    denominator = context_counts[context] + (alpha * vocab_size)
    return numerator / denominator

In [16]:
n = 2
ngram_counts, context_counts = build_ngram_counts(tokinized_sentences, n=n)
print(f"Number of bigrams: {len(ngram_counts)}")
print(f"Number of contexts: {len(context_counts)}")


Number of bigrams: 2281351
Number of contexts: 161367


# predict token & Generate text

In [ ]:
def predict_next_token(
    context_tokens,
    ngram_counts,
    context_counts,
    vocab,
    n=2,
    alpha=1.0,
    top_k=5
):
    """
    Given a list of context tokens, predict the next token using the n-gram model.
    Returns the top_k predictions as (token, probability).
    """
    context = tuple(context_tokens[-(n-1):])
    candidates = []
    
    for word in vocab:
        ngram = context + (word,)
        prob = laplace_probability(ngram, ngram_counts, context_counts, len(vocab), alpha)
        candidates.append((word, prob))
    
    return sorted(candidates, key=lambda x: x[1], reverse=True)[:top_k]


def generate_text_with_limit(
    start_tokens,
    ngram_counts,
    context_counts,
    vocab,
    n=2,
    alpha=1.0,
    max_length=20
):

    generated = start_tokens.copy()
    
    while len(generated) < max_length:
        next_predictions = predict_next_token(
            generated, ngram_counts, context_counts, vocab, n, alpha
        )
        
        if not next_predictions:
            break
            
        next_token = next_predictions[0][0]
        
        if next_token == '</s>':
            break
            
        generated.append(next_token)
        
    return generated

context = ["i", "love"]
generated_seq = generate_text_with_limit(
    start_tokens=context,
    ngram_counts=ngram_counts,
    context_counts=context_counts,
    vocab=vocab,
    n=2,
    alpha=1.0,
    max_length=128
)

print("Generated Sequence:", generated_seq)


Generated Sequence: ['i', 'love', 'with', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot', 'of', 'the', 'film', 'is', 'a', 'lot']


# Calculate Perplexity

In [18]:
def calculate_perplexity(
    tokenized_sentences,
    ngram_counts,
    context_counts,
    vocab_size,
    n=2,
    alpha=1.0
):
    """
    Calculate perplexity using natural logarithm and e.
    Uses log-space computations to avoid numerical underflow.
    """
    total_log_prob = 0.0
    total_tokens = 0
    
    for sentence in tokenized_sentences:
        padded = pad_sentence(sentence, n)
        total_tokens += len(sentence)  # Only count actual tokens, not padding
        
        for i in range(len(padded) - n + 1):
            ngram = tuple(padded[i:i+n])
            prob = laplace_probability(ngram, ngram_counts, context_counts, vocab_size, alpha)
            if prob > 0:  # Avoid log(0)
                total_log_prob += math.log(prob)
            else:
                total_log_prob += float('-inf')  # Handle zero probabilities
    
    # Calculate perplexity: exp(-1/N * sum(log(P)))
    avg_log_prob = total_log_prob / total_tokens if total_tokens > 0 else float('-inf')
    return exp(-avg_log_prob)

# **Analysis**

In [19]:
def evaluate_ngram_model(n, train_sentences, test_sentences, vocab, alpha=1.0):
    """
    Evaluates an n-gram model by training on train_sentences and computing perplexity on test_sentences.
    """
    print(f"\nEvaluating {n}-gram model:")
    
    # Build n-gram model on training data
    train_tokenized = tokinize(train_sentences, vocab)
    ngram_counts, context_counts = build_ngram_counts(train_tokenized, n)
    
    # Evaluate on test data
    test_tokenized = tokinize(test_sentences, vocab)
    perplexity = calculate_perplexity(
        test_tokenized, 
        ngram_counts, 
        context_counts, 
        len(vocab), 
        n, 
        alpha
    )
    
    print(f"Number of unique {n}-grams: {len(ngram_counts)}")
    print(f"Test set perplexity: {perplexity:.2f}")
    
    # Generate a sample text
    context = ["i", "love"] if n > 1 else ["i"]
    generated = generate_text_with_limit(
        context,
        ngram_counts,
        context_counts,
        vocab,
        n,
        alpha,
        max_length=15
    )
    print(f"Sample generated text: {' '.join(generated)}")
    
    return perplexity

# Evaluate models with different n values
alphas = [0.01, 0.5, 1.0, 5.0, 10.0]
n_values = [1, 2, 3]

results = []
for n in n_values:
    for alpha in alphas:
        perp = evaluate_ngram_model(n, train_sentences, test_sentences, vocab, alpha)
        results.append((n, alpha, perp))

# Print comparison table
print("\nModel Comparison:")
print("N-gram | Alpha | Perplexity")
print("-" * 30)
for n, alpha, perp in results:
    print(f"{n:6d} | {alpha:5.2f} | {perp:10.2f}")


Evaluating 1-gram model:
Number of unique 1-grams: 161367
Test set perplexity: 1310.98
Sample generated text: i muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane

Evaluating 1-gram model:
Number of unique 1-grams: 161367
Test set perplexity: 1267.13
Sample generated text: i muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane

Evaluating 1-gram model:
Number of unique 1-grams: 161367
Test set perplexity: 1263.73
Sample generated text: i muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane

Evaluating 1-gram model:
Number of unique 1-grams: 161367
Test set perplexity: 1292.26
Sample generated text: i muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane muzane

Evaluating 1-gram model:
Number of unique 1-grams: 161367
Test set perplexity: 1349.90
Sample generated text: i muzane muzane muzane muzane muzane muzane m

Perplexity Analysis:

1. Unigram Model (n=1)
Most stable across different alpha values
Perplexity range: 1263-1350
Best performance with α=1.0 (perplexity = 1263.73)
Shows limited variation with smoothing

2. Bigram Model (n=2)
Significant variation with alpha values
Best performance with α=0.01 (perplexity = 677.95)
Performance degrades heavily with higher alpha values
Shows better overall performance than unigrams with optimal smoothing

3. Trigram Model (n=3)
Highest sensitivity to smoothing parameter
Much higher perplexity values overall
Best performance with α=0.01 (perplexity = 6073.39)

Text Generation Quality:

1. Unigram: Poor quality
2. Bigram: Better but repetitive ("film is a lot of")
3. Trigram: Most natural-sounding but still repetitive